# BossAssistant — Multi-Department RAG with Knowledge Graph + Routing

**Assignment:** Build a `BossAssistant` that routes HR and Finance questions to department-specific RAG pipelines.


## 1. Environment Setup

In [20]:
!pip install --quiet \
    langchain \
    langchain-community \
    langchain-nvidia-ai-endpoints \
    langchain-huggingface \
    langchain-neo4j \
    neo4j \
    tiktoken \
    sentence-transformers
print('Install done!')

Install done!


### API Keys

You need these in Colab `userdata`:

| Secret | Where to get it |
|---|---|
| `NVIDIA_API_KEY` | https://build.nvidia.com → generate `nvapi-...` key |
| `NEO4J_URI` | https://neo4j.com/cloud/aura-free → create instance → copy URI |
| `NEO4J_USERNAME` | Neo4j Aura (usually `neo4j`) |
| `NEO4J_PASSWORD` | Neo4j Aura (shown once — save it!) |
| `LANGCHAIN_API_KEY` | https://smith.langchain.com → Settings |

In [24]:
import os
from google.colab import userdata

os.environ["NVIDIA_API_KEY"] = userdata.get("NVIDIA_API_KEY")

os.environ["NEO4J_URI"] = userdata.get("NEO4J_URI")
os.environ["NEO4J_USERNAME"] = userdata.get("NEO4J_USERNAME")
os.environ["NEO4J_PASSWORD"] = userdata.get("NEO4J_PASSWORD")

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = userdata.get("LANGCHAIN_PROJECT")

print("Environment ready.")

Environment ready.


## 2. Initialize Models
in this RAG pipeline, i will use qwen/qwen3-next-80b-a3b-instruct for text-to-text generation, nvidia/nv-embedqa-e5-v5 for the embedding

In [25]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings

llm = ChatNVIDIA(
    model="qwen/qwen3-next-80b-a3b-instruct",
    temperature=0,
    max_tokens=1024,
)

embedder = NVIDIAEmbeddings(model="nvidia/nv-embedqa-e5-v5", truncate="END")

# Smoke test both
print("LLM:", llm.invoke("Reply with exactly: OK").content)
print(f"Embedding dim: {len(embedder.embed_query('test'))}")

/usr/local/lib/python3.12/dist-packages/langchain_nvidia_ai_endpoints/_common.py:196: UserWarning: Found qwen/qwen3-next-80b-a3b-instruct in available_models, but type is unknown and inference may fail.
  warnings.warn(


LLM: OK
Embedding dim: 1024


## 3. Synthetic Department Documents (8 each)

In [ ]:
from langchain_core.documents import Document

hr_docs = [
    Document(
        page_content="""Paid Time Off (PTO) Policy. All full-time employees accrue 15 days of paid vacation per year during their first three years of employment. After three years, accrual increases to 20 days per year. PTO is accrued monthly at 1.25 days per month (or 1.67 days for tenured employees). Unused PTO can be carried over up to a maximum of 5 days into the next calendar year. Any excess is forfeited on January 1st. PTO requests must be submitted through Workday at least two weeks in advance for requests longer than 3 consecutive days.""",
        metadata={"source": "HR-001", "title": "PTO Policy", "department": "hr"},
    ),
    Document(
        page_content="""Parental Leave Policy. Birthing parents are eligible for 12 weeks of fully paid parental leave. Non-birthing parents (including adoptive and foster parents) receive 8 weeks of fully paid leave. Leave must be taken within 12 months of the child's birth or adoption. Employees must notify HR at least 30 days before intended leave start date when foreseeable. During parental leave, health insurance benefits continue without interruption and the employee's position is protected.""",
        metadata={"source": "HR-002", "title": "Parental Leave", "department": "hr"},
    ),
    Document(
        page_content="""Remote Work Policy. Employees may work remotely up to 3 days per week with manager approval. Fully remote arrangements require VP-level approval and are evaluated case-by-case based on role requirements. Remote workers must maintain core working hours of 10 AM to 3 PM in their assigned time zone for meeting availability. The company provides a $500 one-time home office stipend for fully remote employees to purchase equipment. Hybrid employees receive $250.""",
        metadata={"source": "HR-003", "title": "Remote Work", "department": "hr"},
    ),
    Document(
        page_content="""Performance Review Process. Formal performance reviews occur twice per year: mid-year in June and annual in December. Reviews use a 5-point scale: 1 (Below Expectations), 2 (Partially Meets), 3 (Meets Expectations), 4 (Exceeds), 5 (Outstanding). Employees complete a self-assessment two weeks before the review meeting. Merit increases and promotion decisions are based on the annual December review. Employees rated 1 or 2 are placed on a 90-day Performance Improvement Plan (PIP).""",
        metadata={"source": "HR-004", "title": "Performance Reviews", "department": "hr"},
    ),
    Document(
        page_content="""Onboarding Process for New Hires. New employees complete a 30-day onboarding program. Week 1 focuses on orientation, IT setup, and benefits enrollment — employees must enroll in benefits within 30 days of start date. Week 2 covers department-specific training. Weeks 3-4 involve shadowing and initial project assignment. New hires are assigned an onboarding buddy and meet with their manager 1-on-1 weekly during onboarding. A 30-day check-in with HR is mandatory.""",
        metadata={"source": "HR-005", "title": "Onboarding", "department": "hr"},
    ),
    Document(
        page_content="""Code of Conduct and Anti-Harassment. The company maintains a zero-tolerance policy for harassment, discrimination, or retaliation of any kind. Reports can be made to any manager, HR Business Partner, or anonymously through the EthicsPoint hotline at 1-800-555-0199. All reports are investigated within 5 business days. Retaliation against anyone making a good-faith report is itself a terminable offense. Annual anti-harassment training is mandatory for all employees and must be completed by November 30 each year.""",
        metadata={"source": "HR-006", "title": "Code of Conduct", "department": "hr"},
    ),
    Document(
        page_content="""Learning and Development Benefits. Each employee receives an annual Learning & Development budget of $2,000 for approved courses, conferences, certifications, and books. Requests must be pre-approved by the employee's manager. The company also offers tuition reimbursement of up to $5,250 per year for degree programs directly related to the employee's role. Employees must remain with the company for 12 months after course completion or repay a prorated amount. Conference attendance requires VP approval if travel is involved.""",
        metadata={"source": "HR-007", "title": "Learning and Development", "department": "hr"},
    ),
    Document(
        page_content="""Health and Wellness Benefits. The company offers three medical plan tiers: Basic PPO, Premium PPO, and HDHP with HSA. Employee premiums are 20%, 15%, and 10% of total cost respectively. Dental and vision are included at no cost to the employee. The wellness program provides a $50/month gym membership reimbursement, free access to the Headspace meditation app, and an annual $300 wellness stipend for fitness equipment, wearables, or mental health services. Employees can use wellness funds for therapy copays.""",
        metadata={"source": "HR-008", "title": "Health and Wellness", "department": "hr"},
    ),
]

fin_docs = [
    Document(
        page_content="""Expense Reimbursement Policy. Employees can be reimbursed for pre-approved business expenses. All expenses must be submitted via Expensify within 30 days of being incurred. Receipts are required for any single expense over $25. Meals during business travel are reimbursable up to $75 per day for domestic travel and $125 per day for international travel. Alcohol is not reimbursable except at client entertainment events with manager approval. Personal expenses incorrectly charged to company cards must be repaid within 14 days.""",
        metadata={"source": "FIN-001", "title": "Expense Reimbursement", "department": "finance"},
    ),
    Document(
        page_content="""Corporate Travel Policy. Business travel requires manager approval in advance. Flights: economy class for flights under 6 hours; premium economy allowed for flights over 6 hours; business class requires VP approval. Hotels must not exceed $250/night in standard US cities, $350/night in NYC/SF/LA, $400/night internationally. Ground transportation: Uber, Lyft, and rental cars are reimbursable. Rental car class is limited to mid-size. Personal vehicle mileage is reimbursed at the current IRS rate of $0.67 per mile.""",
        metadata={"source": "FIN-002", "title": "Travel Policy", "department": "finance"},
    ),
    Document(
        page_content="""Budget Planning and Approval. The annual budget cycle begins October 1st. Department heads submit proposed budgets by October 31st. Finance reviews and consolidates by November 30th. Final budget approval by the CFO and CEO occurs in the December board meeting. Quarterly budget reviews occur in the first week of each new quarter. Any spending exceeding 110% of an approved line item requires CFO approval. Capital expenditures over $50,000 require board approval regardless of budget status.""",
        metadata={"source": "FIN-003", "title": "Budget Planning", "department": "finance"},
    ),
    Document(
        page_content="""Accounts Payable (AP) Process. Vendor invoices should be submitted to ap@company.com. Standard payment terms are Net 30 from invoice date. Early payment discounts of 2/10 Net 30 are negotiated with strategic vendors. All invoices over $10,000 require dual approval: the budget owner and a Finance manager. Invoices over $100,000 additionally require CFO approval. Vendors must be set up in Coupa with a completed W-9 (domestic) or W-8 (international) before their first invoice can be processed.""",
        metadata={"source": "FIN-004", "title": "Accounts Payable", "department": "finance"},
    ),
    Document(
        page_content="""Revenue Recognition Policy. The company follows ASC 606 for revenue recognition. Subscription revenue is recognized ratably over the contract term. Professional services revenue is recognized as services are delivered based on a percentage-of-completion method. One-time setup fees are deferred and recognized over the expected customer life of 3 years. Any contract modifications require review by the Revenue Operations team. Channel partner commissions are recognized at the point of contract signing, not collection.""",
        metadata={"source": "FIN-005", "title": "Revenue Recognition", "department": "finance"},
    ),
    Document(
        page_content="""Corporate Card Program. Employees in roles requiring frequent business expenses are eligible for a corporate American Express card. Card applications are approved by the employee's manager and Finance. Monthly reconciliation in Expensify is required by the 5th of the following month. Cards not reconciled within 60 days are automatically suspended. Cash advances are not permitted on corporate cards. Lost or stolen cards must be reported immediately to Amex (1-800-528-4800) and the corporate card administrator.""",
        metadata={"source": "FIN-006", "title": "Corporate Card", "department": "finance"},
    ),
    Document(
        page_content="""Financial Reporting and Close. The monthly close process runs from the 1st through the 5th business day of the following month. The Finance team publishes the preliminary P&L by day 6 and the board-ready financial package by day 10. Quarterly financial statements are prepared according to GAAP and reviewed by external auditors (KPMG). The fiscal year ends December 31. Annual audit typically completes in late February with the 10-K filed in March. SOX compliance controls apply to all financial processes.""",
        metadata={"source": "FIN-007", "title": "Financial Reporting", "department": "finance"},
    ),
    Document(
        page_content="""Procurement and Purchase Orders. All purchases over $5,000 require a formal purchase order (PO) created in Coupa before the order is placed. Vendors must be in good standing and properly set up. For purchases between $5,000 and $25,000, the budget owner and direct manager approve. Between $25,000 and $100,000, VP approval is required. Above $100,000, CFO approval is required. Sole-source justifications must accompany any PO that did not go through competitive bidding when value exceeds $50,000.""",
        metadata={"source": "FIN-008", "title": "Procurement", "department": "finance"},
    ),
]

print(f"HR: {len(hr_docs)}  |  Finance: {len(fin_docs)}")